In [2]:
# Celda 1 — Cargar p5.js
from IPython.display import HTML
HTML("""
<script src="https://cdnjs.cloudflare.com/ajax/libs/p5.js/1.9.0/p5.min.js"></script>
<p style="color:#4caf50;font-weight:bold;">✅ p5.js cargado</p>
""")

In [2]:
# Celda 2 — Simulación: Leyes de Newton con objeto Forces
# Basado en: Physics for JS Games — Capítulo 5, páginas 112-117
from IPython.display import HTML

HTML("""
  
<div id="newton-root" style="font-family:'Segoe UI',sans-serif; background:#0d1117; padding:20px; border-radius:14px; display:inline-block; color:#e0e0e0;">

  <!-- Título igual al screenshot -->
  <div style="background:#1c2333; padding:10px 16px; border-radius:8px;
              font-size:15px; font-weight:bold; margin-bottom:12px; color:#c9d1d9;">
    Página 112 a 117 Leyes de Newton — Objeto Forces
  </div>

  <!-- ── CONTROLES ───────────────────────────────────────── -->
  <div style="display:flex; gap:18px; flex-wrap:wrap; margin-bottom:14px; align-items:flex-end;">

    <div>
      <div style="font-size:12px; color:#8b949e; margin-bottom:4px;">Masa (m)</div>
      <input type="range" id="massSlider" min="0.5" max="5" step="0.5" value="1"
             style="width:110px;" oninput="document.getElementById('massVal').innerText=parseFloat(this.value).toFixed(1)+' kg'">
      <span id="massVal" style="font-size:12px; margin-left:6px;">1.0 kg</span>
    </div>

    <div>
      <div style="font-size:12px; color:#8b949e; margin-bottom:4px;">Gravedad (g)</div>
      <input type="range" id="gravSlider" min="0.05" max="0.8" step="0.05" value="0.2"
             style="width:110px;" oninput="document.getElementById('gravVal').innerText=parseFloat(this.value).toFixed(2)">
      <span id="gravVal" style="font-size:12px; margin-left:6px;">0.20</span>
    </div>

    <div>
      <div style="font-size:12px; color:#8b949e; margin-bottom:4px;">Drag (k)</div>
      <input type="range" id="dragSlider" min="0" max="0.1" step="0.005" value="0.02"
             style="width:110px;" oninput="document.getElementById('dragVal').innerText=parseFloat(this.value).toFixed(3)">
      <span id="dragVal" style="font-size:12px; margin-left:6px;">0.020</span>
    </div>

    <div style="display:flex; gap:8px;">
      <button onclick="nwtReset()"
        style="background:#e94560;color:#fff;border:none;padding:7px 16px;border-radius:8px;cursor:pointer;font-size:12px;font-weight:bold;">🔄 Reiniciar</button>
      <button onclick="nwtToggle()" id="nwtPauseBtn"
        style="background:#161b22;color:#fff;border:1px solid #30363d;padding:7px 16px;border-radius:8px;cursor:pointer;font-size:12px;font-weight:bold;">⏸ Pausar</button>
    </div>

    <div style="display:flex; gap:8px; align-items:center;">
      <label style="font-size:12px; color:#8b949e; display:flex; align-items:center; gap:5px; cursor:pointer;">
        <input type="checkbox" id="showVectors" checked> Mostrar vectores
      </label>
      <label style="font-size:12px; color:#8b949e; display:flex; align-items:center; gap:5px; cursor:pointer;">
        <input type="checkbox" id="showTrail" checked> Trayectoria
      </label>
    </div>
  </div>

  <!-- ── CANVAS ──────────────────────────────────────────── -->
  <div id="newton-canvas"></div>

  <!-- ── LEYENDA DE VECTORES ──────────────────────────────── -->
  <div style="display:flex; gap:20px; margin-top:10px; font-size:12px;">
    <span><span style="color:#ff6b6b;">●</span> F_gravedad (mg ↓)</span>
    <span><span style="color:#4ecdc4;">●</span> F_drag (−kv)</span>
    <span><span style="color:#ffe66d;">●</span> F_total (resultante)</span>
    <span><span style="color:#a29bfe;">●</span> Velocidad (v)</span>
  </div>

  <!-- ── PANEL DE DATOS ───────────────────────────────────── -->
  <div id="nwt-data" style="font-family:monospace; font-size:11px; color:#8b949e; margin-top:8px; line-height:1.8;"></div>
</div>

<script>
// ═══════════════════════════════════════════════════════════
//  Vector2D  —  clase auxiliar (equivale a la del libro)
// ═══════════════════════════════════════════════════════════
function Vector2D(x, y) { this.x = x; this.y = y; }
Vector2D.prototype.add = function(v) { return new Vector2D(this.x + v.x, this.y + v.y); };
Vector2D.prototype.addScaled = function(v, s) { return new Vector2D(this.x + v.x*s, this.y + v.y*s); };
Vector2D.prototype.multiply = function(s) { return new Vector2D(this.x*s, this.y*s); };
Vector2D.prototype.mag = function() { return Math.sqrt(this.x*this.x + this.y*this.y); };

// ═══════════════════════════════════════════════════════════
//  Forces  —  objeto del libro (pp. 116-117)
// ═══════════════════════════════════════════════════════════
function Forces() {}

Forces.zeroForce = function() {
  return new Vector2D(0, 0);                      // p.117
};

Forces.constantGravity = function(m, g) {
  return new Vector2D(0, m * g);                  // p.117  F_g = mg hacia abajo
};

Forces.linearDrag = function(k, velo) {
  return velo.multiply(-k);                       // p.117  F_d = -kv
};

// ═══════════════════════════════════════════════════════════
//  Sketch p5.js
// ═══════════════════════════════════════════════════════════
var nwtSketch = function(p) {

  var W = 560, H = 360;
  var dt = 1;                  // timestep por frame
  var radius = 18;
  var paused = false;

  // Estado de la partícula
  var pos, velo, acc;
  var force, fGrav, fDrag;    // fuerzas individuales para visualizar
  var trail = [];
  var bounceCount = 0;

  // Leer sliders
  function cfg() {
    return {
      m: parseFloat(document.getElementById('massSlider').value),
      g: parseFloat(document.getElementById('gravSlider').value),
      k: parseFloat(document.getElementById('dragSlider').value)
    };
  }

  function reset() {
    pos   = new Vector2D(80, 60);
    velo  = new Vector2D(3, 0);
    acc   = new Vector2D(0, 0);
    force = Forces.zeroForce();
    fGrav = Forces.zeroForce();
    fDrag = Forces.zeroForce();
    trail = [];
    bounceCount = 0;
  }

  window.nwtReset  = reset;
  window.nwtToggle = function() {
    paused = !paused;
    document.getElementById('nwtPauseBtn').innerText = paused ? '▶ Reanudar' : '⏸ Pausar';
  };

  // ── Ciclo del libro (p.115) ──────────────────────────────
  function calcForce(c) {
    fGrav = Forces.constantGravity(c.m, c.g);   // F_g = m·g
    fDrag = Forces.linearDrag(c.k, velo);        // F_d = −k·v
    force = fGrav.add(fDrag);                    // resultante
  }

  function updateAccel(c) {
    acc = force.multiply(1 / c.m);               // a = F/m  (p.115)
  }

  function updateVelo() {
    velo = velo.addScaled(acc, dt);              // Δv = a·dt (p.115)
  }

  function moveObject() {
    pos = pos.addScaled(velo, dt);               // Δpos = v·dt (p.115)
  }

  // ── Rebote en paredes ────────────────────────────────────
  function handleBounds() {
    if (pos.y > H - radius) {
      pos.y = H - radius;
      velo.y *= -0.75;
      bounceCount++;
    }
    if (pos.x > W - radius) { pos.x = W - radius; velo.x *= -0.85; }
    if (pos.x < radius)     { pos.x = radius;     velo.x *= -0.85; }
    if (pos.y < radius)     { pos.y = radius;      velo.y *= -0.75; }
  }

  // ── Dibujar flecha de vector ─────────────────────────────
  function drawArrow(ox, oy, vx, vy, col, label, scale) {
    scale = scale || 1;
    var ex = ox + vx * scale;
    var ey = oy + vy * scale;
    var len = Math.sqrt(vx*vx + vy*vy) * scale;
    if (len < 1) return;

    p.stroke(col);
    p.strokeWeight(2.2);
    p.line(ox, oy, ex, ey);

    // Punta de flecha
    var angle = Math.atan2(ey - oy, ex - ox);
    var hs = 9;
    p.fill(col);
    p.noStroke();
    p.push();
    p.translate(ex, ey);
    p.rotate(angle);
    p.triangle(0, 0, -hs, -hs*0.45, -hs, hs*0.45);
    p.pop();

    // Etiqueta
    p.noStroke();
    p.fill(col);
    p.textSize(11);
    p.textAlign(p.LEFT);
    p.text(label, ex + 5, ey + 4);
  }

  // ────────────────────────────────────────────────────────
  p.setup = function() {
    var cnv = p.createCanvas(W, H);
    cnv.parent('newton-canvas');
    p.frameRate(60);
    reset();
  };

  p.draw = function() {
    // Fondo
    p.background(13, 17, 23);

    // Cuadrícula
    p.stroke(255, 255, 255, 12);
    p.strokeWeight(1);
    for (var gx = 0; gx < W; gx += 40) p.line(gx, 0, gx, H);
    for (var gy = 0; gy < H; gy += 40) p.line(0, gy, W, gy);

    // Suelo
    p.stroke(233, 69, 96, 160);
    p.strokeWeight(2);
    p.line(0, H - radius, W, H - radius);

    var c = cfg();

    if (!paused) {
      // ── Ciclo F=ma del libro ────────────────────────────
      calcForce(c);
      updateAccel(c);
      updateVelo();
      moveObject();
      handleBounds();
      // ───────────────────────────────────────────────────

      trail.push({x: pos.x, y: pos.y});
      if (trail.length > 120) trail.shift();
    }

    // Trayectoria
    if (document.getElementById('showTrail').checked) {
      p.noFill();
      for (var i = 1; i < trail.length; i++) {
        var a = p.map(i, 0, trail.length, 0, 200);
        p.stroke(162, 155, 254, a);
        p.strokeWeight(p.map(i, 0, trail.length, 0.5, 2));
        p.line(trail[i-1].x, trail[i-1].y, trail[i].x, trail[i].y);
      }
    }

    // ── Vectores de fuerza ──────────────────────────────────
    if (document.getElementById('showVectors').checked) {
      var sc = 60;   // escala visual de los vectores
      var vsc = 20;

      // Velocidad (morado)
      drawArrow(pos.x, pos.y, velo.x, velo.y,
        p.color(162, 155, 254), 'v', vsc);

      // F_gravedad (rojo)
      drawArrow(pos.x - 20, pos.y, fGrav.x, fGrav.y,
        p.color(255, 107, 107), 'Fg', sc);

      // F_drag (cian)
      drawArrow(pos.x, pos.y, fDrag.x, fDrag.y,
        p.color(78, 205, 196), 'Fd', sc);

      // F_total / resultante (amarillo)
      drawArrow(pos.x + 20, pos.y, force.x, force.y,
        p.color(255, 230, 109), 'F', sc);
    }

    // ── Partícula ───────────────────────────────────────────
    // Sombra
    p.noStroke(); p.fill(0, 0, 0, 70);
    p.ellipse(pos.x + 4, pos.y + 4, radius*2, radius*2);

    // Cuerpo
    p.stroke(200, 200, 255, 180); p.strokeWeight(2);
    p.fill(100, 160, 255);
    p.ellipse(pos.x, pos.y, radius*2, radius*2);

    // Brillo
    p.noStroke(); p.fill(255, 255, 255, 110);
    p.ellipse(pos.x - radius*0.3, pos.y - radius*0.35, radius*0.55, radius*0.38);

    // ── Panel de datos ──────────────────────────────────────
    var spd = velo.mag().toFixed(2);
    var fmag = force.mag().toFixed(3);
    document.getElementById('nwt-data').innerHTML =
      '<b style="color:#e0e0e0;">Partícula:</b> ' +
      'pos = (' + pos.x.toFixed(1) + ', ' + pos.y.toFixed(1) + ')' +
      ' &nbsp;|&nbsp; ' +
      'v = (' + velo.x.toFixed(2) + ', ' + velo.y.toFixed(2) + ')  |speed| = ' + spd +
      '<br>' +
      '<b style="color:#ff6b6b;">F_grav</b> = (0, ' + fGrav.y.toFixed(3) + ')' +
      ' &nbsp;|&nbsp; ' +
      '<b style="color:#4ecdc4;">F_drag</b> = (' + fDrag.x.toFixed(3) + ', ' + fDrag.y.toFixed(3) + ')' +
      ' &nbsp;|&nbsp; ' +
      '<b style="color:#ffe66d;">F_total</b> = (' + force.x.toFixed(3) + ', ' + force.y.toFixed(3) + ')  |F| = ' + fmag +
      '<br>' +
      '<b style="color:#a29bfe;">a</b> = (' + acc.x.toFixed(4) + ', ' + acc.y.toFixed(4) + ')' +
      ' &nbsp;|&nbsp; m = ' + c.m.toFixed(1) + ' kg' +
      ' &nbsp;|&nbsp; rebotes = ' + bounceCount;

    // FPS
    p.fill(255,255,255,60); p.noStroke(); p.textSize(10);
    p.textAlign(p.RIGHT);
    p.text('FPS: ' + Math.round(p.frameRate()), W - 6, 14);
  };
};

new p5(nwtSketch);
</script>
""")